# Simulación del dogma central de la biología molecular

Este cuaderno explica la teoría de cada etapa y ejecuta el código real (`io_utils.py`, `replicacion.py`, `transcripcion.py`, `traduccion.py`) para ver cómo se transforma la información desde el ADN hasta la proteína.

| Etapa | Proceso | Molde | Producto |
|---|---|---|---|
| 1 | Replicación | ADN | ADN |
| 2 | Transcripción | ADN | ARNm |
| 3 | Traducción | ARNm | Proteína |

## 1. El dogma central

El dogma central de la biología molecular describe cómo fluye la información genética dentro de la célula:

ADN -- replicación --> ADN -- transcripción --> ARNm -- traducción --> proteína

- **Replicación:** el ADN se copia antes de que la célula se divida, para que cada célula hija tenga el suyo.
- **Transcripción:** un gen del ADN se copia en una molécula de ARN mensajero (ARNm).
- **Traducción:** el ribosoma lee el ARNm de tres en tres letras (codones) y une los aminoácidos correspondientes para formar la proteína.

El ADN y el ARN se parecen, pero se diferencian en tres cosas: timina frente a uracilo, doble hélice frente a una sola hebra, y desoxirribosa frente a ribosa.

## 2. Datos de entrada

El simulador lee secuencias de ADN en dos formatos, que `io_utils.py` distingue por la extensión:

- FASTA: una línea de cabecera que empieza por `>` (con el identificador) y después la secuencia.
- RAW: solo la secuencia, sin cabecera.

Trabajamos con dos secuencias:

- `EJEMPLO`: una secuencia inventada de 51 nucleótidos.
- `NM_000207.3`: una secuencia real de NCBI, el ARNm de la insulina humana.

`NM_000207.3` es el código con el que NCBI identifica esa secuencia. Aunque se trata de un ARNm, la secuencia aparece escrita con T en lugar de U. Para poder utilizarla como entrada de nuestro simulador, la tomamos como representación de la hebra codificante de ADN. Esta hebra tiene la misma secuencia que el ARNm, con la única diferencia de que el ADN utiliza T y el ARN utiliza U.

In [1]:
from io_utils import cargar_secuencia

seq_id, insulina = cargar_secuencia("data/insulina_NM_000207.3.fasta")
print(f"{seq_id}: {len(insulina)} nucleótidos")
print(insulina[:60], "...")

NM_000207.3 Homo sapiens insulin (INS), transcript variant 1, mRNA: 465 nucleótidos
AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA ...


In [2]:
EJEMPLO = "TTGACATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGCTTAACG"
print(f"EJEMPLO: {len(EJEMPLO)} nucleótidos")

EJEMPLO: 51 nucleótidos


## 3. Complementariedad

Las bases del ADN se emparejan siempre igual (**complementariedad**): la A con la T y la G con la C. Al transcribir, el ARN se forma leyendo el molde con estas reglas: A → U, T → A, C → G, G → C.

Además:

- Cada hebra tiene un extremo 5' y otro 3'. Las dos hebras de la doble hélice son antiparalelas, es decir, una va de 5' a 3' y la otra de 3' a 5'.
- Las polimerasas solo añaden nucleótidos en sentido 5' → 3'.
- La hebra codificante tiene la misma secuencia que el ARNm (cambiando T por U). La hebra molde es la complementaria y es la que lee la ARN polimerasa.

In [3]:
from Bio.Seq import Seq
from replicacion import complementaria

molde = complementaria(EJEMPLO)
print("Codificante 5'", EJEMPLO, "3'")
print("Molde       3'", molde, "5'")

# El molde escrito 3'->5' y leído al revés (5'->3') es el complemento inverso de Biopython
print("¿Coincide con Biopython?", molde[::-1] == str(Seq(EJEMPLO).reverse_complement()))

Codificante 5' TTGACATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGCTTAACG 3'
Molde       3' AACTGTACCGGTAACATTACCCGGCGACTTTCCCACGGGCTATCGAATTGC 5'
¿Coincide con Biopython? True


## 4. Replicación del ADN

Antes de que una célula se divida, copia todo su ADN. Eso es la replicación. Es semiconservativa porque cada molécula hija conserva una hebra de la molécula original (la hebra parental) y recibe una hebra nueva.

### Qué ocurre

1. **Se abre la doble hélice.** La topoisomerasa alivia la tensión que se genera al abrir la hélice y la helicasa separa las dos hebras, rompiendo los puentes de hidrógeno entre las bases. El punto donde se separan es la horquilla de replicación.
2. **Cada hebra sirve de molde.** Cada hebra original se usa para fabricar una hebra nueva complementaria (A con T, G con C).
3. **Se colocan los cebadores.** La ADN polimerasa, la enzima que construye el ADN, no puede empezar una hebra desde cero, necesita un trocito de arranque. Ese trocito es el cebador, es de ARN y lo fabrica la primasa.
4. **Se construye la hebra nueva.** La ADN polimerasa añade nucleótidos a partir del cebador, siempre en sentido 5' → 3'.
5. **Se sustituyen los cebadores y se unen los trozos.** Los cebadores de ARN se cambian por ADN y la ligasa une los fragmentos de ADN que han quedado sueltos.

### Cadena líder y cadena rezagada

La ADN polimerasa construye la hebra nueva siempre en sentido 5' → 3', y por eso lee la hebra molde en el sentido contrario, de 3' a 5'. Como las dos hebras del ADN van en sentidos opuestos, solo una de ellas se puede leer en el mismo sentido en que se abre la horquilla. Por eso las dos hebras nuevas no se construyen igual:

- La **cadena líder** se sintetiza de forma **continua**. La polimerasa avanza en el mismo sentido en que se abre la horquilla, así que le basta un único cebador al principio.
- La **cadena rezagada** se sintetiza de forma **discontinua**. Si imaginamos que la horquilla avanza hacia la derecha, aquí la polimerasa solo puede copiar hacia la izquierda, es decir, "hacia atrás" (de derecha a izquierda), y por eso no puede seguir a la helicasa. Cuando se abre un trozo de ADN, la primasa pone un cebador en el extremo derecho de ese trozo, el más cercano a la horquilla, y la polimerasa copia desde ahí hacia la izquierda. Después pasa al siguiente trozo y repite. Cada trozo copiado es un fragmento de Okazaki y al final la ligasa los une todos.

### Simulación

En el simulador, las letras en **minúscula** (como `uugac`) son ARN, es decir, los cebadores, y las **mayúsculas** son ADN. Un cebador de ARN lleva **U** en lugar de T. Para que se vea mejor, los fragmentos de Okazaki son muy cortos (15 nucleótidos); en una célula real son bastante más largos.

In [4]:
from replicacion import mostrar_replicacion

hija_1, hija_2 = mostrar_replicacion(EJEMPLO)


=== REPLICACIÓN DEL ADN ===

1. ADN parental:
  5' TTGACATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGCTTAACG 3'
  3' AACTGTACCGGTAACATTACCCGGCGACTTTCCCACGGGCTATCGAATTGC 5'

2. Apertura de la doble hélice [Topoisomerasa, Helicasa]:
  La topoisomerasa reduce la tensión y la helicasa separa las dos hebras.
  Hebra parental 1: 5' TTGACATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGCTTAACG 3'
  Hebra parental 2: 3' AACTGTACCGGTAACATTACCCGGCGACTTTCCCACGGGCTATCGAATTGC 5'

3. Cadena líder [Primasa, ADN polimerasa]:
  Cebador: uugac
  Nueva hebra: 5' uugacATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGCTTAACG 3'
  Síntesis continua por ADN polimerasa.

4. Cadena rezagada [Primasa, ADN polimerasa]:
  Okazaki 1 [1-15]: cebador guaac
  Okazaki 2 [16-30]: cebador gacuu
  Okazaki 3 [31-45]: cebador uaucg
  Total de fragmentos: 4
  Síntesis discontinua por ADN polimerasa.
  La ADN polimerasa elimina los cebadores y los sustituye por ADN.
  La ligasa une los fragmentos de Okazaki.

5. Moléculas hijas:
  Hija 1:
    5' TTG

**Los fragmentos de Okazaki.** La salida de arriba solo enseña los tres primeros; aquí aparecen todos. La hebra rezagada se copia en trozos de 15 letras y cada trozo tiene su propio cebador de ARN, que ocupa las 5 últimas letras del fragmento (las de la derecha), escritas en minúscula y con U. El último fragmento es más corto porque 51 letras no se reparten en trozos exactos de 15.

In [5]:
from replicacion import sintetizar_rezagada

print("Nº  Posiciones  Fragmento (3'->5')  Cebador de ARN (5')")
for numero, (inicio, fin, cebador, fragmento) in enumerate(sintetizar_rezagada(EJEMPLO), 1):
    rango = f"{inicio}-{fin}"
    print(f"{numero:<4}{rango:<12}{fragmento:<20}{cebador}")

Nº  Posiciones  Fragmento (3'->5')  Cebador de ARN (5')
1   1-15        AACTGTACCGGTAAC     guaac
2   16-30       ATTACCCGGCGACTT     gacuu
3   31-45       TCCCACGGGCTATCG     uaucg
4   46-51       AATTGC              auugc


**Comprobación de la replicación.** Las dos moléculas hijas deben ser iguales a la original. Cada hija se guarda como un par de hebras: la de arriba (5' → 3') y la de abajo (3' → 5'). Aquí comprobamos que, en las dos hijas, la de arriba es la codificante y la de abajo es la molde. Si todo está bien, sale `True`.

In [6]:
from replicacion import replicar_adn

hija_1, hija_2 = replicar_adn(EJEMPLO)
molde = complementaria(EJEMPLO)

print("Hija 1 = líder nueva + molde parental :", hija_1 == (EJEMPLO, molde))
print("Hija 2 = codificante parental + rezagada nueva:", hija_2 == (EJEMPLO, molde))

Hija 1 = líder nueva + molde parental : True
Hija 2 = codificante parental + rezagada nueva: True


## 5. Transcripción

La transcripción es el proceso mediante el cual la información del ADN se copia a una molécula de ARN mensajero (ARNm). La enzima encargada de hacerlo es la ARN polimerasa.

### Qué ocurre

1. **Se abre la doble hélice.** La ARN polimerasa abre una pequeña parte del ADN para poder acceder a sus bases.
2. **Se utiliza una hebra como molde.** La ARN polimerasa lee la hebra molde en sentido 3' → 5'.
3. **Se construye el ARNm.** La ARN polimerasa añade ribonucleótidos y construye el ARNm siempre en sentido 5' → 3'. No necesita un cebador para empezar.
4. **Se siguen las reglas de complementariedad.** Las bases se emparejan así: A → U, T → A, C → G y G → C.
5. **Termina la transcripción.** Cuando se alcanza una señal de terminación, se libera el ARNm.

El ARNm obtenido tiene la misma secuencia que la hebra codificante, cambiando únicamente la T por U:

```text
ADN codificante:  5' ATG GCC ATT 3'
ADN molde:        3' TAC CGG TAA 5'
ARNm:             5' AUG GCC AUU 3'
```

### Simulación

En nuestro simulador, la transcripción utiliza la hebra codificante de la molécula hija obtenida en la replicación (`hija_1[0]`).

De esta forma, los procesos quedan encadenados:

**ADN → ADN → ARNm**

La función `transcribir_adn()` obtiene primero la hebra molde y después genera el ARNm aplicando las reglas de complementariedad.


In [7]:
from transcripcion import mostrar_transcripcion

arnm = mostrar_transcripcion(hija_1[0])


=== TRANSCRIPCIÓN DE ADN A ARNm [ARN polimerasa] ===
  La ARN polimerasa lee la hebra molde (3'->5') y sintetiza el ARNm (5'->3').
  Complementariedad molde -> ARNm:  A->U  T->A  C->G  G->C

  ADN Codificante: 5' TTGACATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAGCTTAACG 3'
  ADN Molde:       3' AACTGTACCGGTAACATTACCCGGCGACTTTCCCACGGGCTATCGAATTGC 5'
  ARNm:            5' UUGACAUGGCCAUUGUAAUGGGCCGCUGAAAGGGUGCCCGAUAGCUUAACG 3'


In [8]:
print("¿Coincide con Biopython?", arnm == str(Seq(EJEMPLO).transcribe()))

¿Coincide con Biopython? True


## 6. Traducción

La **traducción** es el proceso mediante el cual la información del ARN mensajero (ARNm) se utiliza para fabricar una proteína.

### Qué ocurre

1. **El ribosoma se une al ARNm.** El ribosoma lee la secuencia del ARNm de tres en tres bases. Cada grupo de tres bases es un codón.
2. **Comienza la traducción.** El ribosoma busca el codón de inicio **AUG**, que corresponde al aminoácido metionina (Met).
3. **Se incorporan los aminoácidos.** Cada codón del ARNm indica qué aminoácido debe añadirse. Un ARNt lleva ese aminoácido hasta el ribosoma y su anticodón se une al codón correspondiente del ARNm.
4. **Se forma la proteína.** El ribosoma va uniendo los aminoácidos mediante enlaces peptídicos, formando una cadena.
5. **Termina la traducción.** Cuando el ribosoma encuentra un codón de parada (UAA, UAG o UGA), termina la traducción y se libera la proteína.

### Codones y anticodones

El codón se encuentra en el ARNm y el anticodón en el ARNt. Son complementarios y están orientados en sentidos opuestos:

```text
ARNm:  5' AUG 3'
          |||
ARNt:  3' UAC 5'
```

En nuestro simulador, el anticodón se obtiene aplicando las reglas de complementariedad:

**A → U, U → A, C → G, G → C**

### Código genético

Un codón es un grupo de tres bases del ARNm. Hay 64 codones, 61 codifican aminoácidos y 3 son señales de parada (UAA, UAG y UGA). El código es degenerado: hay 61 codones para solo 20 aminoácidos, así que varios codones codifican el mismo aminoácido. AUG es el codón de inicio y codifica metionina.

Los aminoácidos no tienen complemento, es decir, la correspondencia es codón → aminoácido, no una complementariedad entre bases. Los esenciales  no los fabrica el cuerpo humano y deben venir de la dieta, el resto sí.

Una CDS (secuencia codificante) empieza en un codón de inicio, tiene un número entero de codones (longitud múltiplo de 3), termina en un codón de parada y no tiene paradas intermedias. El simulador lee así, empieza en el primer AUG, toma codones completos y se detiene en la primera parada, y avisa si falta la parada o sobran nucleótidos.

A continuación se imprime la tabla de codones:

In [9]:
from traduccion import obtener_tabla

print(obtener_tabla(1))

Table 1 Standard, SGC0

  |  U      |  C      |  A      |  G      |
--+---------+---------+---------+---------+--
U | UUU F   | UCU S   | UAU Y   | UGU C   | U
U | UUC F   | UCC S   | UAC Y   | UGC C   | C
U | UUA L   | UCA S   | UAA Stop| UGA Stop| A
U | UUG L(s)| UCG S   | UAG Stop| UGG W   | G
--+---------+---------+---------+---------+--
C | CUU L   | CCU P   | CAU H   | CGU R   | U
C | CUC L   | CCC P   | CAC H   | CGC R   | C
C | CUA L   | CCA P   | CAA Q   | CGA R   | A
C | CUG L(s)| CCG P   | CAG Q   | CGG R   | G
--+---------+---------+---------+---------+--
A | AUU I   | ACU T   | AAU N   | AGU S   | U
A | AUC I   | ACC T   | AAC N   | AGC S   | C
A | AUA I   | ACA T   | AAA K   | AGA R   | A
A | AUG M(s)| ACG T   | AAG K   | AGG R   | G
--+---------+---------+---------+---------+--
G | GUU V   | GCU A   | GAU D   | GGU G   | U
G | GUC V   | GCC A   | GAC D   | GGC G   | C
G | GUA V   | GCA A   | GAA E   | GGA G   | A
G | GUG V   | GCG A   | GAG E   | GGG G   | G
--+---------

### Simulación

En nuestro simulador, la traducción utiliza el ARNm obtenido en la etapa anterior:

**ADN → ADN → ARNm → proteína**

La función `mostrar_traduccion()` muestra los codones, sus anticodones, los aminoácidos correspondientes y la proteína final.

Para representar la unión entre el ARNm y el ARNt, el simulador calcula el anticodón correspondiente a cada codón.

El anticodón se obtiene aplicando la complementariedad de bases y se muestra en sentido 3' → 5'.

In [10]:
from traduccion import mostrar_traduccion

proteina = mostrar_traduccion(arnm)


=== TRADUCCIÓN DE ARNm A PROTEÍNA [Ribosoma, ARNt] ===
  Inicio: AUG en la posición 6

  Codones y anticodones (anticodón en 3'->5'):
  1. Codón: AUG  Anticodón: UAC  Aminoácido: Met (M)
  2. Codón: GCC  Anticodón: CGG  Aminoácido: Ala (A)
  3. Codón: AUU  Anticodón: UAA  Aminoácido: Ile (I)
  4. Codón: GUA  Anticodón: CAU  Aminoácido: Val (V)
  5. Codón: AUG  Anticodón: UAC  Aminoácido: Met (M)
  6. Codón: GGC  Anticodón: CCG  Aminoácido: Gly (G)
  7. Codón: CGC  Anticodón: GCG  Aminoácido: Arg (R)
  STOP: UGA en la posición 27

  Proteína (7 aa): MAIVMGR
  En tres letras: Met-Ala-Ile-Val-Met-Gly-Arg


Si la longitud no es múltiplo de 3 o falta el codón de parada, el simulador traduce los codones completos y avisa:

In [11]:
_ = mostrar_traduccion("AUGGCCAUUGU")


=== TRADUCCIÓN DE ARNm A PROTEÍNA [Ribosoma, ARNt] ===
  Inicio: AUG en la posición 1

  Codones y anticodones (anticodón en 3'->5'):
  1. Codón: AUG  Anticodón: UAC  Aminoácido: Met (M)
  2. Codón: GCC  Anticodón: CGG  Aminoácido: Ala (A)
  3. Codón: AUU  Anticodón: UAA  Aminoácido: Ile (I)
  AVISO: no hay codón de parada.
  AVISO: la longitud desde el AUG no es múltiplo de 3 (sobran 2 nt).

  Proteína (3 aa): MAI
  En tres letras: Met-Ala-Ile


## 7. Flujo completo con un caso real: la insulina humana

Se ejecuta el recorrido ADN → ADN, ADN → ARN y ARN → proteína con la secuencia `NM_000207.3`. Para no llenar la pantalla, el simulador dibuja solo los primeros 60 nucleótidos, pero **calcula con la secuencia completa**.

In [12]:
hija_1, hija_2 = mostrar_replicacion(insulina)
arnm = mostrar_transcripcion(hija_1[0])
proteina = mostrar_traduccion(arnm)


=== REPLICACIÓN DEL ADN ===

1. ADN parental:
  5' AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA ... 3'
  3' TCGGGAGGTCCTGTCCGACGTAGTCTTCTCCGGTAGTTCGTCTAGTGACAGGAAGACGGT ... 5'

2. Apertura de la doble hélice [Topoisomerasa, Helicasa]:
  La topoisomerasa reduce la tensión y la helicasa separa las dos hebras.
  Hebra parental 1: 5' AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA ... 3'
  Hebra parental 2: 3' TCGGGAGGTCCTGTCCGACGTAGTCTTCTCCGGTAGTTCGTCTAGTGACAGGAAGACGGT ... 5'

3. Cadena líder [Primasa, ADN polimerasa]:
  Cebador: agccc
  Nueva hebra: 5' agcccTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA ... 3'
  Síntesis continua por ADN polimerasa.

4. Cadena rezagada [Primasa, ADN polimerasa]:
  Okazaki 1 [1-15]: cebador cuguc
  Okazaki 2 [16-30]: cebador uucuc
  Okazaki 3 [31-45]: cebador ucuag
  Total de fragmentos: 31
  Síntesis discontinua por ADN polimerasa.
  La ADN polimerasa elimina los cebadores y los sustituye por ADN.
  La ligasa une lo

In [13]:
print("=== RESUMEN DEL FLUJO ===")
print(f"ADN -> ADN       : 2 moléculas hijas idénticas a la original: {hija_1 == hija_2 and hija_1[0] == insulina}")
print(f"ADN -> ARNm      : {len(arnm)} nt, con U en lugar de T")
print(f"ARNm -> proteína : {len(proteina)} aa desde el AUG en la posición {arnm.find('AUG') + 1}")

# Comprobación con Biopython sobre la secuencia completa
desde_aug = Seq(insulina).transcribe()[arnm.find("AUG"):]
esperada = str(desde_aug[:len(desde_aug) // 3 * 3].translate(to_stop=True))
print("¿La proteína coincide con la de Biopython?", proteina == esperada)

=== RESUMEN DEL FLUJO ===
ADN -> ADN       : 2 moléculas hijas idénticas a la original: True
ADN -> ARNm      : 465 nt, con U en lugar de T
ARNm -> proteína : 110 aa desde el AUG en la posición 60
¿La proteína coincide con la de Biopython? True
